E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

In [5]:
import torch
import torch.nn.functional as F

In [4]:
from torch.utils.data.dataset import random_split

In [1]:
# open file
words = open('../names.txt', 'r').read().splitlines()
words[:3]

['emma', 'olivia', 'ava']

In [42]:
# 拆分训练集

def SplitSet(words):
    g = torch.Generator().manual_seed(22)
    n = len(words)
    c_train = int(n * 0.8)
    c_dev = int(n * 0.1)
    c_test = n - c_train - c_dev
    split = random_split(words, [c_train, c_dev, c_test], generator=g)
    return [list(split[0]), list(split[1]), list(split[2])]

n_train, n_dev, n_test = SplitSet(words)

Bigram

In [115]:
class Bigram:

    def __init__(self):
        g = torch.Generator().manual_seed(22)
        self.W = torch.randn((27, 27), generator=g, requires_grad=True)

    def initWrods(self, inWords = []):
        chars = sorted(list(set(''.join(words))))
        self.stoi = {s:i+1 for i,s in enumerate(chars) }
        self.stoi['.'] = 0
        self.itos = {i:s for s,i in self.stoi.items()}

        xs, self.ys = [], []
        for w in words:
            chs = ['.'] + list(w) + ['.']
            for ch1, ch2 in zip(chs, chs[1:]):
                ix1 = self.stoi[ch1]
                ix2 = self.stoi[ch2]
                xs.append(ix1)
                self.ys.append(ix2)
        xs = torch.tensor(xs)
        self. ys = torch.tensor(self.ys)
        self.num = xs.nelement()
        self.xenc = F.one_hot(xs, num_classes=27).float()

    def getLoss(self):
        # xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
        logits = self.xenc @ self.W # predict log-counts
        counts = logits.exp() # counts, equivalent to N
        probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
        loss = -probs[torch.arange(self.num), self.ys].log().mean() + 0.01*(self.W**2).mean()
        return loss
        
    def train(self, inTrainwords):
        self.initWrods(inTrainwords)
        for k in range(100):
            loss = self.getLoss()
            if (k % 10 == 0):
                print(f'loss = {loss}')
            # backward pass
            self.W.grad = None # set to zero the gradient
            loss.backward()
          
            # update
            self.W.data += -70 * self.W.grad
        
    def output(self):
        g = torch.Generator().manual_seed(22)
        for i in range(5):
          out = []
          ix = 0
          while True:
            xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
            logits = xenc @ self.W # predict log-counts
            counts = logits.exp() # counts, equivalent to N
            p = counts / counts.sum(1, keepdims=True) # probabilities for next character
            ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
            out.append(self.itos[ix])
            if ix == 0:
              break
          print(''.join(out))

    def test(self, intestWords):
        self.initWrods(intestWords)
        loss = self.getLoss()
        print(f'test loss = {loss}')
        

In [116]:
bi = Bigram()

In [117]:
bi.train(n_train)

loss = 3.802926778793335
loss = 2.6127328872680664
loss = 2.54245662689209
loss = 2.518622875213623
loss = 2.506434440612793
loss = 2.4991490840911865
loss = 2.494410514831543
loss = 2.4911694526672363
loss = 2.488877058029175
loss = 2.4872121810913086


In [113]:
bi.test(n_test)

test loss = 2.4859728813171387


In [114]:
bi.output()

ffcliydslo.
raieviamanthylesh.
rd.
gprysai.
ynene.


['y', 'n', 'e', 'n', 'e', '.']

Trigram

In [131]:
class Trigram:
    def __init__(self):
        g = torch.Generator().manual_seed(2147483647)
        self.W = torch.randn((729, 27), generator = g, requires_grad = True)
        self.initIndex()

    def initIndex(self):
        chars = '.abcdefghijklmnopqrstuvwxyz'
        self.sti = {} # 前两个字符对应index
        self.its = {} # index 到前两个字符
        i = 0
        for ch1 in chars:
            for ch2 in chars:
                firstTwo = ch1 + ch2
                self.sti[firstTwo] = i
                self.its[i] = firstTwo
                i += 1

        # 单字符对应index
        self.cti = {s:i for i,s in enumerate(chars)}
        self.itc = {i:s for i,s in enumerate(chars)}

    def initWords(self, inWords):
        xs, self.ys = [], []
        for w in inWords:
            chs = ['.'] + ['.'] + list(w) + ['.']
            for i in range(0, len(chs) - 2):
                    tmp = chs[i:i+3]
                    firstTwo = tmp[0] + tmp[1]
                    ch = tmp[2]
                    xs.append(self.sti[firstTwo])
                    self.ys.append(self.cti[ch])
                    # print(f'{firstTwo}, {ch}')
            xs = torch.tensor(xs)
            self.ys = torch.tensor(self.ys)
        self.xenc = F.one_hot(xs, num_classes = 729).float()

    def getLoss(self):
        logits = self.xenc @ self.W # log-counts
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims = True)
        
        #loss
        loss = -probs[torch.arange(self.num), self.ys].log().mean()

    def train(self, inWords):
        self.initWords(inWords)

        for i in range(100):
            loss = self.getLoss()
            if (i % 10 == 0):
                print(f'loss: {loss}')
            self.W.grad = None
            loss.backward()

            self.W += -80 * self.W.grad

    def Test(self, inWords):
        self.initWords(inWords)

        loss = self.getLoss()
        print(f'loss: {loss}')

    def output(self):
        g = torch.Generator().maul_seed(22)
        for i in range(0,3):
            out = '..'
            ix = 0
            while True:
                outxenc = F.one_hot(torch.tensor([ix]), num_classes = 729).float()
                logits = outxenc @ self.W
                counts = logits.exp()
                probs = counts / counts.sum(1, keepdims = True)
                i = torch.multinomial(probs, num_samples = 1, replacement = True, generator = g).item()
                out += self.itc[i]
                ix = self.sti[out[-2:]]
                # print(out[-2:])
                if (i == 0):
                    break
            print(f'{out}')

In [132]:
tri = Trigram()

In [133]:
tri.train(n_train)

AttributeError: 'Tensor' object has no attribute 'append'